# Comprehensive Summary of Files Created in the Audio Processing Chain

Here's a detailed breakdown of each file created by the script, including its location, purpose, and the processing steps involved in its creation:

## 1. Processed Vocals (`1. Processed Vocals.wav`)
- **Location**: `C:\Users\cogpsy-vrlab\Documents\PPS_module\BreathingPilot\AudioInstructionsBoxBreathing\TargetStimuli\Intermediary_Files`
- **Purpose**: Processed vocals track with adjusted volume for better audibility
- **Processing Steps**:
  - Load the original vocals file
  - Convert to 48kHz sample rate for precise timing
  - Adjust volume (multiplied by 1.2) to make speech clearer
  - Save as WAV format for higher quality

## 2. Processed Vocals with Muting (`1b. Processed Vocals with Hold Muting.wav`)
- **Location**: `C:\Users\cogpsy-vrlab\Documents\PPS_module\BreathingPilot\AudioInstructionsBoxBreathing\TargetStimuli\Intermediary_Files`
- **Purpose**: Vocals track with 3-second muted segments after each "hold" instruction
- **Processing Steps**:
  - Take the processed vocals from step 1
  - Load timestamps from CSV file
  - For each timestamp, create a 3-second silent segment
  - Replace those segments in the audio with silence
  - Save as WAV format

## 3. Processed Backing Track (`2. Processed Backing Track.wav`)
- **Location**: `C:\Users\cogpsy-vrlab\Documents\PPS_module\BreathingPilot\AudioInstructionsBoxBreathing\TargetStimuli\Intermediary_Files`
- **Purpose**: Processed backing music with adjusted volume
- **Processing Steps**:
  - Load the original backing track
  - Convert to 48kHz sample rate
  - Adjust volume (multiplied by 0.8) to make it quieter than the vocals
  - Save as WAV format

## 4. Combined Raw Audio (`3. Box Breathing Combined (Raw).wav`)
- **Location**: `C:\Users\cogpsy-vrlab\Documents\PPS_module\BreathingPilot\AudioInstructionsBoxBreathing\TargetStimuli`
- **Purpose**: Combines the muted vocals and backing track into a single audio file
- **Processing Steps**:
  - Take the muted vocals from step 2
  - Take the processed backing track from step 3
  - Overlay them (mix them together)
  - Ensure both tracks are the same length (truncate if needed)
  - Save as WAV format

## 5. Unlooped Reference (`4. Box Breathing Combined (Unlooped).wav`)
- **Location**: `C:\Users\cogpsy-vrlab\Documents\PPS_module\BreathingPilot\AudioInstructionsBoxBreathing\TargetStimuli`
- **Purpose**: Serves as an unlooped reference copy of the combined audio
- **Processing Steps**:
  - Simply create a copy of the combined audio from step 4
  - This serves as a reference for the unmodified audio before extensions

## 6. Loop Segment (`5. Box Breathing Loop Segment.wav`)
- **Location**: `C:\Users\cogpsy-vrlab\Documents\PPS_module\BreathingPilot\AudioInstructionsBoxBreathing\TargetStimuli\Intermediary_Files`
- **Purpose**: The segment of audio that will be repeated to create the extended version
- **Processing Steps**:
  - Extract a segment from the combined audio (step 4)
  - Start from the first "hold" timestamp
  - End at the timestamp before the ending segment plus buffer
  - Save as WAV format for use in creating the extended audio

## 7. Final Extended Audio (`6. Box Breathing Combined (Segment Injected).wav`)
- **Location**: `C:\Users\cogpsy-vrlab\Documents\PPS_module\BreathingPilot\AudioInstructionsBoxBreathing\TargetStimuli`
- **Purpose**: The final extended audio with exactly 220 "hold" events for the experiment
- **Processing Steps**:
  - Start with the intro segment (up to first "hold")
  - Add the main loop segment once
  - Add additional complete loops as needed
  - Add a partial loop if needed (to reach exact number of holds)
  - Attach the ending segment
  - Save as WAV format

## 8. Extended Timestamps CSV (`BoxBreathing_extended_timestamps.csv`)
- **Location**: `C:\Users\cogpsy-vrlab\Documents\PPS_module\BreathingPilot\AudioInstructionsBoxBreathing\TargetStimuli`
- **Purpose**: Contains precise timestamps for all "hold" events in the extended audio
- **Processing Steps**:
  - Calculate new timestamps based on the structure of the extended audio
  - Include original timestamps for intro section
  - Add adjusted timestamps for each loop repetition
  - Add adjusted timestamps for the ending section
  - Save as CSV with columns for formatted time, milliseconds, and sample indices

## 9. Processing Metadata (`processing_metadata.json`)
- **Location**: `C:\Users\cogpsy-vrlab\Documents\PPS_module\BreathingPilot\AudioInstructionsBoxBreathing\TargetStimuli`
- **Purpose**: Documents all processing details and parameters for reproducibility
- **Processing Steps**:
  - Collect information about source files, parameters, and processing steps
  - Compile statistics about hold events and audio properties
  - Document file paths and relationships
  - Save as a structured JSON file

## Processing Workflow Summary

1. First, the script loads timestamps from the CSV file
2. Then it processes the raw vocals track (adjusts volume)
3. Next, it mutes segments after each "hold" timestamp
4. It processes the backing track (adjusts volume)
5. The processed vocals and backing are combined
6. The appropriate segments are extracted for looping
7. An extended version is created by intelligent looping to reach exactly 220 hold events
8. Extended timestamps are calculated and saved
9. Comprehensive metadata is generated

This entire process creates an audio file specifically designed for your peripersonal space experiment, with precise timing of "hold" events and muted segments to allow for your experimental stimuli.

In [ ]:
"""
COMPREHENSIVE BREATHING MEDITATION AUDIO PROCESSING SCRIPT
FOR PERIPERSONAL SPACE EXPERIMENT

This script handles the complete processing chain to prepare audio files for a
peripersonal space measurement experiment using box breathing meditation. Starting
from raw vocals and backing track files, it creates all necessary intermediate and
final output files.

PROJECT GOALS:
- Process raw vocals and backing tracks to create clean base audio
- Mute 3 seconds after each "hold" timestamp from the CSV file
- Create a properly timed and extended box breathing meditation audio
- Generate exactly the specified number of 'hold' breath moments for trials
- Create precise timestamps for each 'hold' event for experimental synchronization
- Support audio-tactile stimuli presentation for peripersonal space measurement
- Provide all necessary intermediary files for reference and troubleshooting

EXPERIMENT DESIGN:
- 220 total trials measuring peripersonal space across breathing conditions
- Includes trials during inhalation, exhalation, and baseline breathing
- Each trial presents audio-tactile stimuli with different SOA values
- The breathing exercise provides precise timing for trial presentation
- The Woojer vibration strap will be synchronized with audio events

COMPLETE PROCESSING CHAIN:
1. Load source vocals and backing track files
2. Load timestamps from CSV file
3. Mute 3 seconds after each timestamp from the CSV file
4. Prepare base audio files with properly timed muting
5. Combine processed vocals and backing track with precise volume balance
6. Extract segments for looping and ending reference
7. Calculate required loops needed to achieve target trial count
8. Construct extended audio with proper looping and ending
9. Generate timestamps for all hold events
10. Export all final files with complete metadata
"""

import os
from pydub import AudioSegment
import pandas as pd
import math
import datetime
import json
import shutil

########## USER CONFIGURATION ##########
# Set the exact number of hold events you want in the final audio
TARGET_HOLD_EVENTS = 220

# Define source file paths
VOCALS_SOURCE_PATH = r"C:\Users\cogpsy-vrlab\Documents\PPS_module\BreathingPilot\AudioInstructionsBoxBreathing\TargetStimuli\1. Box Breathing Relaxation Exercise ｜ 5 Minutes Beginner Pace ｜ Anxiety Reduction Pranayama Technique [oN8xV3Kb5-Q] (Vocals) (MDX v2 Voc FT).mp3"
BACKING_SOURCE_PATH = r"C:\Users\cogpsy-vrlab\Documents\PPS_module\BreathingPilot\AudioInstructionsBoxBreathing\TargetStimuli\2. Box Breathing Relaxation Exercise ｜ 5 Minutes Beginner Pace ｜ Anxiety Reduction Pranayama Technique [oN8xV3Kb5-Q] (Backing Track) (MDX v2 Voc FT) (1).mp3"
TIMESTAMPS_CSV_PATH = r"C:\Users\cogpsy-vrlab\Documents\PPS_module\BreathingPilot\AudioInstructionsBoxBreathing\TargetStimuli\holds.csv"

# Define output directory
OUTPUT_DIR = r"C:\Users\cogpsy-vrlab\Documents\PPS_module\BreathingPilot\AudioInstructionsBoxBreathing\TargetStimuli"

# Define important timestamps
ENDING_TIMESTAMP = "05:12.600"  # Where the ending reference should start

# Audio processing parameters
VOCALS_VOLUME = 1.2       # Volume multiplier for vocals (1.0 = original volume)
BACKING_VOLUME = 0.8      # Volume multiplier for backing track (0.8 = 80% of original volume)
SAMPLE_RATE = 48000       # Sample rate in Hz for output files
BUFFER_AFTER_HOLD = 8000  # Buffer time in ms after hold events (for smooth transitions)
MUTE_DURATION_SECONDS = 3.0  # Duration in seconds to mute after each hold timestamp
#########################################

# Define output file paths (derived from configuration)
INTERMEDIARY_DIR = os.path.join(OUTPUT_DIR, "Intermediary_Files")

# Intermediate file paths (numbered in processing order)
PROCESSED_VOCALS_PATH = os.path.join(INTERMEDIARY_DIR, "1. Processed Vocals.wav")
PROCESSED_VOCALS_MUTED_PATH = os.path.join(INTERMEDIARY_DIR, "1b. Processed Vocals with Hold Muting.wav")
PROCESSED_BACKING_PATH = os.path.join(INTERMEDIARY_DIR, "2. Processed Backing Track.wav")
COMBINED_AUDIO_PATH = os.path.join(OUTPUT_DIR, "3. Box Breathing Combined (Raw).wav")
COMBINED_SELF_COUNTED_PATH = os.path.join(OUTPUT_DIR, "4. Box Breathing Combined (Unlooped).wav")
EXTENDED_LOOP_SEGMENT_PATH = os.path.join(INTERMEDIARY_DIR, "5. Box Breathing Loop Segment.wav")
FINAL_OUTPUT_PATH = os.path.join(OUTPUT_DIR, "6. Box Breathing Combined (Segment Injected).wav")
TIMESTAMPS_OUTPUT_PATH = os.path.join(OUTPUT_DIR, "BoxBreathing_extended_timestamps.csv")
METADATA_PATH = os.path.join(OUTPUT_DIR, "processing_metadata.json")

# Function to convert time string to milliseconds
def time_to_ms(time_str):
    minutes, seconds = time_str.split(':')
    return int(minutes) * 60 * 1000 + float(seconds) * 1000

# Function to convert milliseconds to time string
def ms_to_time(ms):
    total_seconds = ms / 1000
    minutes = int(total_seconds // 60)
    seconds = total_seconds % 60
    return f"{minutes:02d}:{seconds:06.3f}"

# Function to load timestamps from CSV
def load_timestamps_from_csv(csv_path):
    print(f"Loading timestamps from CSV: {csv_path}")
    try:
        df = pd.read_csv(csv_path)
        print(f"Loaded {len(df)} timestamps from CSV")
        
        # Extract timestamps and convert to milliseconds if needed
        timestamps_ms = []
        
        # Check if we have 'milliseconds' column directly
        if 'milliseconds' in df.columns:
            timestamps_ms = df['milliseconds'].tolist()
        # Otherwise use 'timestamp' column
        elif 'timestamp' in df.columns:
            timestamps_ms = [time_to_ms(ts) for ts in df['timestamp'].tolist()]
        else:
            # Try to use the first column if no standard column names found
            first_col = df.columns[0]
            if df[first_col].dtype == 'object':  # Assume string timestamps
                timestamps_ms = [time_to_ms(ts) for ts in df[first_col].tolist()]
            else:  # Assume numeric milliseconds
                timestamps_ms = df[first_col].tolist()
        
        # Create new DataFrame with standardized timestamps
        timestamps_df = pd.DataFrame({
            'timestamp': [ms_to_time(ms) for ms in timestamps_ms],
            'milliseconds': timestamps_ms,
            'sample_index': [int(ms * SAMPLE_RATE / 1000) for ms in timestamps_ms]
        })
        
        return timestamps_df
    except Exception as e:
        print(f"Error loading timestamps from CSV: {str(e)}")
        return None

# Function to process vocals track with volume adjustment
def process_vocals_track(input_path, volume_multiplier=1.0):
    # Load the vocals track
    print(f"Loading vocals track: {input_path}")
    vocals = AudioSegment.from_mp3(input_path)
    vocals = vocals.set_frame_rate(SAMPLE_RATE)  # Ensure consistent sample rate
    
    print(f"Vocals track duration: {vocals.duration_seconds:.2f} seconds")
    
    # Adjust volume
    if volume_multiplier != 1.0:
        vocals = vocals + (20 * math.log10(volume_multiplier))  # Convert to dB
        print(f"Adjusted vocals volume by factor of {volume_multiplier}")
    
    # Save the processed vocals
    vocals.export(PROCESSED_VOCALS_PATH, format="wav")
    print(f"Saved processed vocals track: {PROCESSED_VOCALS_PATH}")
    
    return vocals

# Function to mute segments after hold timestamps
def mute_after_hold_timestamps(vocals_track, timestamps_df, mute_duration_sec):
    """Mute segments after each hold timestamp for specified duration."""
    print(f"Muting {mute_duration_sec} seconds after each hold timestamp...")
    
    # Convert mute duration to milliseconds
    mute_duration_ms = int(mute_duration_sec * 1000)
    
    # Get the timestamps in milliseconds
    timestamps_ms = timestamps_df['milliseconds'].tolist()
    
    # Create a copy of the vocals track
    modified_vocals = vocals_track
    
    # Apply muting to each segment
    for i, start_ms in enumerate(timestamps_ms):
        # Calculate end of mute segment
        end_ms = start_ms + mute_duration_ms
        
        # Ensure we don't exceed the track length
        if start_ms >= len(modified_vocals):
            continue
        if end_ms > len(modified_vocals):
            end_ms = len(modified_vocals)
        
        # Create a silent segment of the same duration
        silence = AudioSegment.silent(duration=end_ms - start_ms)
        
        # Split the audio and insert silence
        before = modified_vocals[:start_ms]
        after = modified_vocals[end_ms:]
        
        # Combine with silence
        modified_vocals = before + silence + after
        
        # Print progress periodically
        if (i+1) % 10 == 0 or i == 0 or i == len(timestamps_ms)-1:
            print(f"Muted segment {i+1}/{len(timestamps_ms)}: {ms_to_time(start_ms)} to {ms_to_time(end_ms)}")
    
    # Save the processed vocals with muting
    modified_vocals.export(PROCESSED_VOCALS_MUTED_PATH, format="wav")
    print(f"Saved vocals track with muted segments: {PROCESSED_VOCALS_MUTED_PATH}")
    
    return modified_vocals

# Function to process backing track
def process_backing_track(input_path, volume_multiplier=1.0):
    # Load the backing track
    print(f"Loading backing track: {input_path}")
    backing = AudioSegment.from_mp3(input_path)
    backing = backing.set_frame_rate(SAMPLE_RATE)  # Ensure consistent sample rate
    
    print(f"Backing track duration: {backing.duration_seconds:.2f} seconds")
    
    # Adjust volume
    if volume_multiplier != 1.0:
        backing = backing + (20 * math.log10(volume_multiplier))  # Convert to dB
        print(f"Adjusted backing track volume by factor of {volume_multiplier}")
    
    # Save the processed backing track
    backing.export(PROCESSED_BACKING_PATH, format="wav")
    print(f"Saved processed backing track: {PROCESSED_BACKING_PATH}")
    
    return backing

# Function to combine vocals and backing track
def combine_audio_tracks(vocals, backing):
    # Ensure both tracks are the same length
    if len(vocals) > len(backing):
        print(f"Vocals track is longer than backing track. Truncating vocals to match backing track length.")
        combined = vocals[:len(backing)].overlay(backing)
    elif len(backing) > len(vocals):
        print(f"Backing track is longer than vocals track. Truncating backing to match vocals length.")
        combined = vocals.overlay(backing[:len(vocals)])
    else:
        combined = vocals.overlay(backing)
    
    print(f"Combined audio duration: {combined.duration_seconds:.2f} seconds")
    
    # Save the combined audio
    combined.export(COMBINED_AUDIO_PATH, format="wav")
    print(f"Saved combined audio: {COMBINED_AUDIO_PATH}")
    
    # Create a copy for the unlooped version
    combined.export(COMBINED_SELF_COUNTED_PATH, format="wav")
    print(f"Saved unlooped reference copy: {COMBINED_SELF_COUNTED_PATH}")
    
    return combined

# Function to create extended audio file with exact number of hold events
def create_extended_audio(base_audio, timestamps_df, ending_timestamp, target_hold_count):
    # Convert ending timestamp to milliseconds
    ending_ms = time_to_ms(ending_timestamp)
    
    # Get timestamps in milliseconds
    timestamps_ms = timestamps_df['milliseconds'].tolist()
    
    # Get total number of hold events in original file
    total_original_holds = len(timestamps_ms)
    print(f"Original file contains {total_original_holds} hold events")
    
    # Check if we already have enough hold events
    if total_original_holds >= target_hold_count:
        print(f"Original file already has {total_original_holds} hold events, which meets the target of {target_hold_count}")
        print("No extension needed.")
        return base_audio, timestamps_df, {
            "complete_loops_added": 0,
            "partial_loop_holds_added": 0,
            "total_holds": total_original_holds
        }
    
    # Calculate additional hold events needed
    additional_holds_needed = target_hold_count - total_original_holds
    
    # Define the loopable section
    first_hold_ms = timestamps_ms[0]
    
    # Find the timestamp just before ending
    last_hold_before_ending_ms = 0
    for ms in timestamps_ms:
        if ms < ending_ms and ms > last_hold_before_ending_ms:
            last_hold_before_ending_ms = ms
    
    # Add buffer for smooth transition
    loop_section_end_ms = last_hold_before_ending_ms + BUFFER_AFTER_HOLD
    
    # Make sure we don't exceed the audio length
    loop_section_end_ms = min(loop_section_end_ms, len(base_audio))
    
    # Extract the ending from the base audio
    ending_segment = base_audio[ending_ms:]
    print(f"Extracted ending segment starting at {ending_timestamp}")
    print(f"Ending segment duration: {ending_segment.duration_seconds:.2f} seconds")
    
    # Calculate holds in ending and before ending
    holds_in_ending = sum(1 for ms in timestamps_ms if ms >= ending_ms)
    holds_before_ending = total_original_holds - holds_in_ending
    
    print(f"Holds before ending: {holds_before_ending}")
    print(f"Holds in ending section: {holds_in_ending}")
    print(f"Need {additional_holds_needed} additional hold events")
    
    # Extract loop segment
    loop_segment = base_audio[first_hold_ms:loop_section_end_ms]
    print(f"Loop segment duration: {loop_segment.duration_seconds:.2f} seconds")
    
    # Save loop segment for reference
    loop_segment.export(EXTENDED_LOOP_SEGMENT_PATH, format="wav")
    print(f"Saved loop segment: {EXTENDED_LOOP_SEGMENT_PATH}")
    
    # Calculate holds in a complete loop segment
    loop_holds = sum(1 for ms in timestamps_ms if first_hold_ms <= ms < loop_section_end_ms)
    print(f"A complete loop segment contains {loop_holds} hold events")
    
    # Calculate how many complete loops we need and how many holds from the final partial loop
    complete_loops_needed = additional_holds_needed // loop_holds
    remaining_holds_needed = additional_holds_needed % loop_holds
    
    print(f"Need {complete_loops_needed} complete loops plus {remaining_holds_needed} additional hold events")
    
    # Create a list of holds in the loop segment with their timestamps
    loop_holds_list = [(i, ms) for i, ms in enumerate(timestamps_ms) if first_hold_ms <= ms < loop_section_end_ms]
    
    # Calculate the cutoff point for the partial final loop if needed
    if remaining_holds_needed > 0:
        # Get the timestamp of the last hold we need in the partial loop
        partial_loop_end_idx = remaining_holds_needed - 1
        if partial_loop_end_idx < len(loop_holds_list):
            _, partial_loop_end_ts = loop_holds_list[partial_loop_end_idx]
            # Add buffer for smooth transition
            partial_loop_end_ms = partial_loop_end_ts + BUFFER_AFTER_HOLD
            # Ensure we don't go beyond the loop segment
            partial_loop_end_ms = min(partial_loop_end_ms, loop_section_end_ms)
        else:
            # Fallback if calculation is off
            partial_loop_end_ms = loop_section_end_ms
        
        partial_loop_segment = base_audio[first_hold_ms:partial_loop_end_ms]
        print(f"Partial loop segment duration: {partial_loop_segment.duration_seconds:.2f} seconds")
        print(f"Partial loop contains {remaining_holds_needed} hold events")
    else:
        partial_loop_segment = AudioSegment.empty()
        partial_loop_end_ms = first_hold_ms  # No partial loop needed
    
    # Construct the extended audio
    # Start with the audio up to the first hold (intro)
    extended_audio = base_audio[:first_hold_ms]
    print(f"Starting with intro audio up to first hold, duration: {extended_audio.duration_seconds:.2f} seconds")
    
    # Add first loop segment
    extended_audio += loop_segment
    print(f"Added initial loop segment, duration now: {extended_audio.duration_seconds:.2f} seconds")
    
    # Add additional complete loops
    for i in range(complete_loops_needed):
        extended_audio += loop_segment
        print(f"Added complete loop {i+1}/{complete_loops_needed}, current duration: {extended_audio.duration_seconds:.2f} seconds")
    
    # Add partial loop if needed
    if remaining_holds_needed > 0:
        extended_audio += partial_loop_segment
        print(f"Added partial loop with {remaining_holds_needed} hold events, current duration: {extended_audio.duration_seconds:.2f} seconds")
    
    # Attach the ending segment
    extended_audio += ending_segment
    print(f"Attached ending segment, final duration: {extended_audio.duration_seconds:.2f} seconds")
    
    # Generate extended timestamps list
    extended_timestamps_ms = []
    
    # Track the current offset for adding timestamps
    current_offset_ms = 0
    
    # Add timestamps for the intro section (up to first hold)
    for ms in timestamps_ms:
        if ms < first_hold_ms:
            extended_timestamps_ms.append(ms)
    
    # Add timestamps for the first loop segment
    for ms in timestamps_ms:
        if first_hold_ms <= ms < loop_section_end_ms:
            new_ts = current_offset_ms + (ms - first_hold_ms) + first_hold_ms
            extended_timestamps_ms.append(new_ts)
    
    # Update offset after first loop
    current_offset_ms = first_hold_ms + (loop_section_end_ms - first_hold_ms)
    
    # Add timestamps for each additional complete loop
    loop_duration_ms = loop_section_end_ms - first_hold_ms
    
    for i in range(complete_loops_needed):
        for ms in timestamps_ms:
            if first_hold_ms <= ms < loop_section_end_ms:
                new_ts = current_offset_ms + (ms - first_hold_ms)
                extended_timestamps_ms.append(new_ts)
        current_offset_ms += loop_duration_ms
    
    # Add timestamps for the partial loop
    if remaining_holds_needed > 0:
        count = 0
        for ms in timestamps_ms:
            if first_hold_ms <= ms < partial_loop_end_ms:
                new_ts = current_offset_ms + (ms - first_hold_ms)
                extended_timestamps_ms.append(new_ts)
                count += 1
                if count >= remaining_holds_needed:
                    break
        current_offset_ms += partial_loop_end_ms - first_hold_ms
    
    # Add timestamps for the ending segment
    ending_offset_ms = current_offset_ms
    
    for ms in timestamps_ms:
        if ms >= ending_ms:
            new_ts = ending_offset_ms + (ms - ending_ms)
            extended_timestamps_ms.append(new_ts)
    
    # Sort timestamps
    extended_timestamps_ms.sort()
    
    # Convert to time strings and calculate sample indices
    extended_timestamps = [ms_to_time(ms) for ms in extended_timestamps_ms]
    sample_indices = [int(ms * SAMPLE_RATE / 1000) for ms in extended_timestamps_ms]
    
    # Create extended timestamps DataFrame
    extended_timestamps_df = pd.DataFrame({
        'timestamp': extended_timestamps,
        'milliseconds': extended_timestamps_ms,
        'sample_index': sample_indices
    })
    
    # Verify we have the exact number of hold events
    actual_holds = len(extended_timestamps_ms)
    print(f"Target hold events: {target_hold_count}")
    print(f"Actual hold events: {actual_holds}")
    
    if actual_holds != target_hold_count:
        print(f"WARNING: Actual hold events ({actual_holds}) does not match target ({target_hold_count})")
        # Adjust by truncating if we have too many
        if actual_holds > target_hold_count:
            extra = actual_holds - target_hold_count
            print(f"Truncating {extra} excess hold events to match target")
            extended_timestamps_df = extended_timestamps_df.iloc[:target_hold_count]
            actual_holds = target_hold_count
    
    return extended_audio, extended_timestamps_df, {
        "complete_loops_added": complete_loops_needed,
        "partial_loop_holds_added": remaining_holds_needed,
        "total_holds": actual_holds
    }

def main():
    try:
        # Print header
        print("\n" + "="*80)
        print("COMPREHENSIVE BREATHING MEDITATION AUDIO PROCESSING")
        print("FOR PERIPERSONAL SPACE EXPERIMENT")
        print("="*80)
        print(f"TARGET: {TARGET_HOLD_EVENTS} hold events for experimental trials")
        print("="*80 + "\n")
        
        # Create intermediary directory if it doesn't exist
        if not os.path.exists(INTERMEDIARY_DIR):
            os.makedirs(INTERMEDIARY_DIR)
            print(f"Created intermediary files directory: {INTERMEDIARY_DIR}")
        
        # STEP 1: Load timestamps from CSV
        print("\n--- STEP 1: LOADING TIMESTAMPS FROM CSV ---")
        timestamps_df = load_timestamps_from_csv(TIMESTAMPS_CSV_PATH)
        if timestamps_df is None:
            print("Error loading timestamps. Cannot proceed.")
            return
        
        # STEP 2: Process the vocals track
        print("\n--- STEP 2: PROCESSING VOCALS TRACK ---")
        processed_vocals = process_vocals_track(VOCALS_SOURCE_PATH, VOCALS_VOLUME)
        
        # STEP 3: Mute segments after each hold timestamp
        print("\n--- STEP 3: MUTING SEGMENTS AFTER HOLD TIMESTAMPS ---")
        processed_vocals_with_muting = mute_after_hold_timestamps(processed_vocals, timestamps_df, MUTE_DURATION_SECONDS)
        
        # STEP 4: Process the backing track
        print("\n--- STEP 4: PROCESSING BACKING TRACK ---")
        processed_backing = process_backing_track(BACKING_SOURCE_PATH, BACKING_VOLUME)
        
        # STEP 5: Combine vocals and backing track
        print("\n--- STEP 5: COMBINING VOCALS AND BACKING TRACK ---")
        combined_audio = combine_audio_tracks(processed_vocals_with_muting, processed_backing)
        
        # STEP 6: Create extended audio with exact number of hold events
        print("\n--- STEP 6: CREATING EXTENDED AUDIO ---")
        extended_audio, extended_timestamps_df, extension_info = create_extended_audio(
            combined_audio, timestamps_df, ENDING_TIMESTAMP, TARGET_HOLD_EVENTS
        )
        
        # STEP 7: Export final extended audio
        print("\n--- STEP 7: EXPORTING FINAL AUDIO ---")
        print(f"Exporting extended audio to {FINAL_OUTPUT_PATH}...")
        extended_audio.export(FINAL_OUTPUT_PATH, format="wav")
        
        # Verify the final file
        if os.path.exists(FINAL_OUTPUT_PATH):
            final_size = os.path.getsize(FINAL_OUTPUT_PATH) / (1024 * 1024)  # Size in MB
            print(f"Final file created successfully: {final_size:.2f} MB")
            
            try:
                final_audio = AudioSegment.from_wav(FINAL_OUTPUT_PATH)
                print(f"Final audio file verified: {final_audio.duration_seconds:.2f} seconds ({final_audio.duration_seconds/60:.2f} minutes)")
            except Exception as e:
                print(f"Warning: Could not verify final file: {str(e)}")
        else:
            print(f"ERROR: Final file not found at {FINAL_OUTPUT_PATH}")
        
        # STEP 8: Save extended timestamps
        print("\n--- STEP 8: SAVING EXTENDED TIMESTAMPS ---")
        extended_timestamps_df.to_csv(TIMESTAMPS_OUTPUT_PATH, index=False)
        print(f"CSV with {len(extended_timestamps_df)} hold timestamps created: {TIMESTAMPS_OUTPUT_PATH}")
        
        # STEP 9: Create detailed metadata
        print("\n--- STEP 9: CREATING METADATA ---")
        metadata = {
            "processing_date": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "source_files": {
                "vocals_track": VOCALS_SOURCE_PATH,
                "backing_track": BACKING_SOURCE_PATH,
                "timestamps_csv": TIMESTAMPS_CSV_PATH
            },
            "processing_parameters": {
                "vocals_volume": VOCALS_VOLUME,
                "backing_volume": BACKING_VOLUME,
                "sample_rate": SAMPLE_RATE,
                "mute_duration_seconds": MUTE_DURATION_SECONDS,
                "ending_timestamp": ENDING_TIMESTAMP
            },
            "output_files": {
                "processed_vocals": PROCESSED_VOCALS_PATH,
                "processed_vocals_with_muting": PROCESSED_VOCALS_MUTED_PATH,
                "processed_backing": PROCESSED_BACKING_PATH,
                "combined_raw": COMBINED_AUDIO_PATH,
                "unlooped_reference": COMBINED_SELF_COUNTED_PATH,
                "loop_segment": EXTENDED_LOOP_SEGMENT_PATH,
                "final_output": FINAL_OUTPUT_PATH,
                "timestamps": TIMESTAMPS_OUTPUT_PATH
            },
            "extension_info": {
                "original_hold_events": len(timestamps_df),
                "target_hold_events": TARGET_HOLD_EVENTS,
                "actual_hold_events": extension_info["total_holds"],
                "complete_loops_added": extension_info["complete_loops_added"],
                "partial_loop_holds_added": extension_info["partial_loop_holds_added"]
            },
            "audio_properties": {
                "final_duration_seconds": extended_audio.duration_seconds,
                "final_duration_minutes": extended_audio.duration_seconds / 60,
                "final_size_mb": final_size
            },
            "processing_steps": [
                "1. Load timestamps from CSV",
                "2. Process vocals track (adjust volume)",
                "3. Mute segments after hold timestamps",
                "4. Process backing track (adjust volume)",
                "5. Combine vocals and backing track",
                "6. Create extended audio with exact number of hold events",
                "7. Export final extended audio",
                "8. Save extended timestamps CSV",
                "9. Create detailed metadata"
            ]
        }
        
        with open(METADATA_PATH, 'w') as f:
            json.dump(metadata, f, indent=2)
        
        print(f"Processing metadata saved: {METADATA_PATH}")
        
        # Print processing summary
        print("\n" + "="*80)
        print("PROCESSING SUMMARY:")
        print("="*80)
        print(f"  Source vocals track: {os.path.basename(VOCALS_SOURCE_PATH)}")
        print(f"  Source backing track: {os.path.basename(BACKING_SOURCE_PATH)}")
        print(f"  Source timestamps: {os.path.basename(TIMESTAMPS_CSV_PATH)}")
        print(f"  Vocals processed: Yes (volume: {VOCALS_VOLUME}x)")
        print(f"  Segments muted: Yes ({MUTE_DURATION_SECONDS}s after each hold timestamp)")
        print(f"  Original hold events: {len(timestamps_df)}")
        print(f"  Target hold events: {TARGET_HOLD_EVENTS}")
        print(f"  Actual hold events: {extension_info['total_holds']}")
        print(f"  Complete loops added: {extension_info['complete_loops_added']}")
        print(f"  Partial loop with {extension_info['partial_loop_holds_added']} hold events added: {'Yes' if extension_info['partial_loop_holds_added'] > 0 else 'No'}")
        print(f"  Final audio duration: {extended_audio.duration_seconds:.2f} seconds ({extended_audio.duration_seconds/60:.2f} minutes)")
        print("\nOutput files created:")
        print(f"  1. Processed vocals: {os.path.basename(PROCESSED_VOCALS_PATH)}")
        print(f"  2. Processed vocals with muting: {os.path.basename(PROCESSED_VOCALS_MUTED_PATH)}")
        print(f"  3. Processed backing: {os.path.basename(PROCESSED_BACKING_PATH)}")
        print(f"  4. Combined raw audio: {os.path.basename(COMBINED_AUDIO_PATH)}")
        print(f"  5. Unlooped reference: {os.path.basename(COMBINED_SELF_COUNTED_PATH)}")
        print(f"  6. Loop segment: {os.path.basename(EXTENDED_LOOP_SEGMENT_PATH)}")
        print(f"  7. Final extended audio: {os.path.basename(FINAL_OUTPUT_PATH)}")
        print(f"  8. Timestamps CSV: {os.path.basename(TIMESTAMPS_OUTPUT_PATH)}")
        print(f"  9. Processing metadata: {os.path.basename(METADATA_PATH)}")
        print("="*80)
        print("\nPROCESSING COMPLETE")
        print("All files have been created successfully.")
        print("The audio file is now ready for use in your peripersonal space experiment.")
        print("Use the timestamps CSV to synchronize your experimental stimuli.")
        
    except Exception as e:
        print(f"\nERROR: {str(e)}")
        import traceback
        traceback.print_exc()
        print("\nPlease check your input files and FFmpeg installation.")

if __name__ == "__main__":
    main()